# Phase 1 (Obj B Baseline): FE + IV Core Models

This notebook implements the baseline model layer for Obj B: panel FE and IV-TWFE estimates for inflation and GDP growth.

## Design choices

- Core controls: trade_open, pop_growth, investment_share.
- GDP models exclude gdp_pc_growth to avoid mechanical overlap risk.
- IV diagnostics are read from linearmodels first-stage diagnostics.
- Interpretation should separate relevance (chi2 significance) from strong-IV status.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS
from statsmodels.tsa.filters.hp_filter import hpfilter

warnings.filterwarnings("ignore")


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "02_data/analysis_ready/macro_growth_merged.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root with analysis-ready data.")


ROOT = find_project_root(Path.cwd().resolve())
PANEL_PATH = ROOT / "02_data/analysis_ready/macro_growth_merged.csv"
CONTROLS_PATH = ROOT / "02_data/supporting/phase1_controls.csv"
INSTRUMENTS_PATH = ROOT / "02_data/supporting/phase1_instruments.csv"
OUT_DIR = ROOT / "03_analysis_notebooks/exports/phase1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESTRICTED_CONTROLS = ["trade_open", "pop_growth", "investment_share"]
CHI2_1_95_CRITICAL = 3.841458820694124
STRONG_IV_STAT_THRESHOLD = 10.0
ROOT

PosixPath('/Users/stevenchung/Desktop/P12B_File/Monetary_Panel')

In [2]:
base = pd.read_csv(PANEL_PATH)
controls = pd.read_csv(CONTROLS_PATH)
instruments = pd.read_csv(INSTRUMENTS_PATH)

panel = (
    base.merge(controls, on=["Country Name", "year"], how="left")
        .merge(instruments, on=["Country Name", "year"], how="left")
        .rename(columns={"Country Name": "country"})
        .sort_values(["country", "year"])
        .copy()
)


def build_output_gap(df: pd.DataFrame) -> pd.Series:
    out = pd.Series(np.nan, index=df.index, dtype=float)
    for _, idx in df.groupby("country").groups.items():
        sub = df.loc[idx].sort_values("year")
        growth = sub["gdp_growth"].astype(float)
        if growth.notna().sum() < 10:
            continue
        level = np.log1p(growth.fillna(0.0)).cumsum()
        cycle, _ = hpfilter(level, lamb=6.25)
        out.loc[sub.index] = 100.0 * cycle
    return out


panel["output_gap_hp"] = build_output_gap(panel)

pd.DataFrame({
    "metric": ["rows", "countries", "year_min", "year_max", "output_gap_non_missing"],
    "value": [
        len(panel),
        panel["country"].nunique(),
        int(panel["year"].min()),
        int(panel["year"].max()),
        int(panel["output_gap_hp"].notna().sum()),
    ],
})

,metric,value
0,rows,4750
1,countries,160
2,year_min,1991
3,year_max,2024
4,output_gap_non_missing,4750


In [3]:
def run_fe(df, outcome, controls=None):
    controls = controls or []
    cols = ["country", "year", outcome, "m2_growth", *controls]
    fit_data = df[cols].dropna().set_index(["country", "year"]).sort_index()
    rhs = "m2_growth" + (" + " + " + ".join(controls) if controls else "")
    formula = f"{outcome} ~ 1 + {rhs} + EntityEffects + TimeEffects"
    return PanelOLS.from_formula(formula, data=fit_data).fit(cov_type="clustered", cluster_entity=True)

fe_infl_base = run_fe(panel, "inflation", controls=[])
fe_gdp_base = run_fe(panel, "gdp_growth", controls=[])
fe_infl_ctrl = run_fe(panel, "inflation", controls=RESTRICTED_CONTROLS)
fe_gdp_ctrl = run_fe(panel, "gdp_growth", controls=RESTRICTED_CONTROLS)

fe_table = pd.DataFrame([
    {"model": "fe_baseline", "outcome": "inflation", "coef": float(fe_infl_base.params["m2_growth"]), "p_value": float(fe_infl_base.pvalues["m2_growth"]), "nobs": int(fe_infl_base.nobs)},
    {"model": "fe_baseline", "outcome": "gdp_growth", "coef": float(fe_gdp_base.params["m2_growth"]), "p_value": float(fe_gdp_base.pvalues["m2_growth"]), "nobs": int(fe_gdp_base.nobs)},
    {"model": "fe_controls", "outcome": "inflation", "coef": float(fe_infl_ctrl.params["m2_growth"]), "p_value": float(fe_infl_ctrl.pvalues["m2_growth"]), "nobs": int(fe_infl_ctrl.nobs)},
    {"model": "fe_controls", "outcome": "gdp_growth", "coef": float(fe_gdp_ctrl.params["m2_growth"]), "p_value": float(fe_gdp_ctrl.pvalues["m2_growth"]), "nobs": int(fe_gdp_ctrl.nobs)},
])
fe_table

,model,outcome,coef,p_value,nobs
0,fe_baseline,inflation,0.664851,0.000175,4750
1,fe_baseline,gdp_growth,-0.005980,0.563932,4750
2,fe_controls,inflation,0.602236,0.003634,3511
3,fe_controls,gdp_growth,-0.005556,0.586802,3511


In [4]:
def run_iv_twfe(df, outcome, instrument, controls):
    cols = ["country", "year", outcome, "m2_growth", instrument, *controls]
    fit_data = df[cols].dropna().copy()

    rhs = " + ".join(controls)
    if rhs:
        formula = f"{outcome} ~ 1 + {rhs} + C(country) + C(year) [m2_growth ~ {instrument}]"
    else:
        formula = f"{outcome} ~ 1 + C(country) + C(year) [m2_growth ~ {instrument}]"

    res = IV2SLS.from_formula(formula, data=fit_data).fit(cov_type="clustered", clusters=fit_data["country"])
    fs = res.first_stage.diagnostics.loc["m2_growth"]
    return {
        "outcome": outcome,
        "instrument": instrument,
        "coef": float(res.params["m2_growth"]),
        "p_value": float(res.pvalues["m2_growth"]),
        "nobs": int(res.nobs),
        "first_stage_stat": float(fs["f.stat"]),
        "first_stage_p": float(fs["f.pval"]),
        "partial_rsquared": float(fs["partial.rsquared"]),
    }

iv_rows = []
for outcome in ["inflation", "gdp_growth"]:
    for instrument in ["instrument_m2_external_level", "instrument_m2_l1"]:
        iv_rows.append(run_iv_twfe(panel, outcome, instrument, RESTRICTED_CONTROLS))

iv_table = pd.DataFrame(iv_rows).sort_values(["outcome", "instrument"])
iv_table

,outcome,instrument,coef,p_value,nobs,first_stage_stat,first_stage_p,partial_rsquared
2,gdp_growth,instrument_m2_external_level,-0.186431,1.519834e-01,3511,5.586925,0.018095,0.001602
3,gdp_growth,instrument_m2_l1,-0.023591,4.031274e-01,3388,5.701932,0.016946,0.084120
0,inflation,instrument_m2_external_level,1.468079,5.002413e-05,3511,5.586925,0.018095,0.001602
1,inflation,instrument_m2_l1,1.578427,2.717454e-08,3388,5.701932,0.016946,0.084120


In [5]:
preferred = iv_table[(iv_table["outcome"] == "inflation") & (iv_table["instrument"] == "instrument_m2_external_level")].iloc[0]
gate_table = pd.DataFrame([
    {"criterion": "first_stage_relevance_chi2_95", "value": float(preferred["first_stage_stat"]), "threshold": f">{CHI2_1_95_CRITICAL:.4f}", "pass": bool(preferred["first_stage_stat"] > CHI2_1_95_CRITICAL)},
    {"criterion": "first_stage_p_lt_0p05", "value": float(preferred["first_stage_p"]), "threshold": "<0.05", "pass": bool(preferred["first_stage_p"] < 0.05)},
    {"criterion": "first_stage_strong_iv_ge_10", "value": float(preferred["first_stage_stat"]), "threshold": f">={STRONG_IV_STAT_THRESHOLD:.1f}", "pass": bool(preferred["first_stage_stat"] >= STRONG_IV_STAT_THRESHOLD)},
])

fe_table.to_csv(OUT_DIR / "phase1_fe_results.csv", index=False)
iv_table.to_csv(OUT_DIR / "phase1_iv_results.csv", index=False)
gate_table.to_csv(OUT_DIR / "phase1_gate_read.csv", index=False)

display(gate_table)
print("Saved Phase 1 outputs to:", OUT_DIR)

,criterion,value,threshold,pass
0,first_stage_relevance_chi2_95,5.586925,>3.8415,True
1,first_stage_p_lt_0p05,0.018095,<0.05,True
2,first_stage_strong_iv_ge_10,5.586925,>=10.0,False


Saved Phase 1 outputs to: /Users/stevenchung/Desktop/P12B_File/Monetary_Panel/03_analysis_notebooks/exports/phase1


In [6]:
# Phillips-curve block (Obj B / YoY): in-sample TWFE + 2016-2020 holdout forecast.
# Spec 1: pi_t = a_i + d_t + rho * pi_{t-1} + beta * gap_t + eps          (baseline Phillips)
# Spec 2: pi_t = a_i + d_t + rho * pi_{t-1} + beta * gap_t + gamma * m2g  (augmented Phillips)
# Forecast: country-FE only (no year FE so we can predict unseen years), train <=2015, holdout 2016-2020.

phil_df = panel[["country", "year", "inflation", "output_gap_hp", "m2_growth"]].copy()
phil_df = phil_df.sort_values(["country", "year"])
phil_df["inflation_l1"] = phil_df.groupby("country")["inflation"].shift(1)
phil_df = phil_df.dropna(subset=["inflation", "inflation_l1", "output_gap_hp", "m2_growth"])
phil_panel = phil_df.set_index(["country", "year"])

phil_base = PanelOLS.from_formula(
    "inflation ~ 1 + inflation_l1 + output_gap_hp + EntityEffects + TimeEffects",
    data=phil_panel,
).fit(cov_type="clustered", cluster_entity=True, cluster_time=True)

phil_aug = PanelOLS.from_formula(
    "inflation ~ 1 + inflation_l1 + output_gap_hp + m2_growth + EntityEffects + TimeEffects",
    data=phil_panel,
).fit(cov_type="clustered", cluster_entity=True, cluster_time=True)


def _coef_row(name, res):
    row = {"model": name, "nobs": int(res.nobs), "within_r2": float(res.rsquared_within)}
    for v in ["inflation_l1", "output_gap_hp", "m2_growth"]:
        row[f"coef_{v}"] = float(res.params[v]) if v in res.params.index else np.nan
        row[f"p_{v}"] = float(res.pvalues[v]) if v in res.pvalues.index else np.nan
    return row


phillips_table = pd.DataFrame([
    _coef_row("phillips_baseline", phil_base),
    _coef_row("phillips_augmented", phil_aug),
])
phillips_table.to_csv(OUT_DIR / "phase1_phillips_results.csv", index=False)

# --- Inflation forecast: country-FE specs, train <= 2015, holdout 2016-2020 ---
TRAIN_END = 2015


def _fit_country_fe(train_df, x_cols, y_col="inflation"):
    grp_means = train_df.groupby("country")[x_cols + [y_col]].transform("mean")
    Xd = train_df[x_cols].values - grp_means[x_cols].values
    yd = train_df[y_col].values - grp_means[y_col].values
    beta, *_ = np.linalg.lstsq(Xd, yd, rcond=None)
    cmean = train_df.groupby("country")[x_cols + [y_col]].mean()
    cmean["alpha"] = cmean[y_col] - cmean[x_cols].values @ beta
    return beta, cmean["alpha"]


def _predict(test_df, x_cols, beta, alpha):
    test_df = test_df.copy()
    test_df["alpha_i"] = test_df["country"].map(alpha)
    test_df = test_df.dropna(subset=["alpha_i"])
    test_df["pred"] = test_df["alpha_i"].values + test_df[x_cols].values @ beta
    return test_df


phil_train = phil_df[phil_df["year"] <= TRAIN_END]
phil_test = phil_df[phil_df["year"] > TRAIN_END]
forecast_specs = {
    "naive_ar1":          ["inflation_l1"],
    "phillips":           ["inflation_l1", "output_gap_hp"],
    "phillips_augmented": ["inflation_l1", "output_gap_hp", "m2_growth"],
}

forecast_rows, pred_frames = [], []
for spec_name, xcols in forecast_specs.items():
    beta, alpha = _fit_country_fe(phil_train, xcols)
    pred_df = _predict(phil_test, xcols, beta, alpha)
    err = pred_df["pred"] - pred_df["inflation"]
    forecast_rows.append({
        "model": spec_name,
        "rmse": float(np.sqrt((err ** 2).mean())),
        "mae": float(err.abs().mean()),
        "bias": float(err.mean()),
        "n_holdout": int(len(pred_df)),
        "n_countries_holdout": int(pred_df["country"].nunique()),
        "train_end_year": TRAIN_END,
    })
    pred_frames.append(pred_df[["country", "year", "inflation", "pred"]].assign(model=spec_name))

forecast_skill = pd.DataFrame(forecast_rows)
forecast_skill.to_csv(OUT_DIR / "phase1_phillips_forecast_skill.csv", index=False)

forecast_predictions = pd.concat(pred_frames, ignore_index=True)
forecast_predictions.to_csv(OUT_DIR / "phase1_phillips_forecast_predictions.csv", index=False)

print("Phillips in-sample (TWFE):")
display(phillips_table)
print(f"\nHoldout forecast skill (train <= {TRAIN_END}, test = {TRAIN_END + 1}-{int(phil_df['year'].max())}):")
display(forecast_skill)

Phillips in-sample (TWFE):


,model,nobs,within_r2,coef_inflation_l1,p_inflation_l1,coef_output_gap_hp,p_output_gap_hp,coef_m2_growth,p_m2_growth
0,phillips_baseline,4590,0.547777,0.592799,5.400770e-07,-0.000896,0.306952,NaN,NaN
1,phillips_augmented,4590,0.671679,0.444962,1.816762e-06,-0.001471,0.080645,0.348591,0.032001



Holdout forecast skill (train <= 2015, test = 2016-2024):


,model,rmse,mae,bias,n_holdout,n_countries_holdout,train_end_year
0,naive_ar1,0.068427,0.030155,0.000104,1216,153,2015
1,phillips,0.068355,0.030237,0.000054,1216,153,2015
2,phillips_augmented,0.063341,0.036061,-0.011388,1216,153,2015


## Interpretation for Obj B baseline

- Use FE and IV coefficient signs/magnitudes as baseline evidence.
- Keep causal language constrained unless first-stage strength is comfortably high.
- Continue to Phase 2 for short-run LP-IV dynamics.

## Obj B Robustness — Clean sample (max annual inflation < 40%)

The full-sample TWFE slope of 0.665 may be inflated by hyperinflation episodes (post-Soviet, Latin American, 1991-1999).
Running the same TWFE on the clean sample (123 countries) tests whether the short-run pass-through survives without those outliers.

In [7]:
from linearmodels.panel import PanelOLS as _PanelOLS

# ── TWFE: clean sample robustness ────────────────────────────────────────────
clean_b = panel[panel['sample_low_inflation'] == 1].dropna(subset=['inflation','m2_growth']).copy()
clean_b_panel = clean_b.set_index(['country','year'])

mod_clean = _PanelOLS.from_formula(
    'inflation ~ m2_growth + EntityEffects + TimeEffects',
    data=clean_b_panel
)
res_clean = mod_clean.fit(cov_type='clustered', cluster_entity=True)

print('TWFE — Clean sample (max inflation < 40%)')
print(f'  n rows:     {len(clean_b)}')
print(f'  n countries:{clean_b["country"].nunique()}')
print(f'  coef:       {res_clean.params["m2_growth"]:.4f}')
print(f'  p-value:    {res_clean.pvalues["m2_growth"]:.5f}')
print(f'  within R2:  {res_clean.rsquared_within:.4f}')
print()
print('Interpretation: if slope collapses near zero, the full-sample short-run result')
print('is driven by hyperinflation episodes, not the modern monetary transmission.')
print('This is the QE-era finding: for non-hyperinflationary economies, year-to-year')
print('money growth barely moves inflation.')

TWFE — Clean sample (max inflation < 40%)
  n rows:     3639
  n countries:123
  coef:       0.0403
  p-value:    0.06212
  within R2:  0.0292

Interpretation: if slope collapses near zero, the full-sample short-run result
is driven by hyperinflation episodes, not the modern monetary transmission.
This is the QE-era finding: for non-hyperinflationary economies, year-to-year
money growth barely moves inflation.


In [8]:
# Cross-sectional dependence test (Pesaran 2004)
# Tests H0: errors are cross-sectionally independent
# With 160 countries and common global shocks (GFC, COVID), CD is expected
import numpy as np
from scipy import stats

def pesaran_cd_test(df, y_col):
    """Pesaran (2004) CD statistic for cross-sectional dependence."""
    countries = df['country'].unique()
    N = len(countries)
    resids = {}

    # Get demeaned residuals per country
    for c in countries:
        sub = df[df['country'] == c].dropna(subset=[y_col, 'm2_growth'])
        if len(sub) < 5:
            continue
        y = sub[y_col].values
        resids[c] = (sub[['year', y_col]].copy().assign(y_dm=y - y.mean())
                     .rename(columns={'y_dm': 'resid'}))

    countries_valid = list(resids.keys())
    N_valid = len(countries_valid)

    # Compute pairwise correlations
    cd_sum = 0.0
    pairs = 0
    for i in range(N_valid):
        for j in range(i+1, N_valid):
            ci, cj = countries_valid[i], countries_valid[j]
            ri = resids[ci][['year','resid']]
            rj = resids[cj][['year','resid']]
            merged = ri.merge(rj, on='year', suffixes=('_i','_j'))
            if len(merged) < 5:
                continue
            yi = merged['resid_i'].values
            yj = merged['resid_j'].values
            if yi.std() == 0 or yj.std() == 0:
                continue
            rho_ij = np.corrcoef(yi, yj)[0, 1]
            Tij = len(merged)
            cd_sum += np.sqrt(Tij) * rho_ij
            pairs += 1

    CD = np.sqrt(2 / (N_valid * (N_valid - 1))) * cd_sum
    p_val = 2 * (1 - stats.norm.cdf(abs(CD)))
    print(f"Pesaran CD test: CD = {CD:.3f}, p = {p_val:.4f}")
    print(f"  N countries = {N_valid}, pairs used = {pairs}")
    if p_val < 0.05:
        print("  REJECT H0: cross-sectional dependence detected")
    else:
        print("  Cannot reject H0: no strong CSD evidence")
    return CD, p_val

# Run on inflation residuals
CD_stat, CD_p = pesaran_cd_test(panel, 'inflation')


Pesaran CD test: CD = 158.435, p = 0.0000
  N countries = 160, pairs used = 12652
  REJECT H0: cross-sectional dependence detected


In [9]:
# Panel unit root: individual ADF tests (Im-Pesaran-Shin approach)
# H0: all panels contain unit root
# If we reject, series are stationary (I(0)) — TWFE is valid
import numpy as np
from scipy import stats
from statsmodels.tsa.stattools import adfuller

def ips_unit_root(df, col, min_obs=10):
    """Im-Pesaran-Shin (2003) panel unit root test via individual ADF."""
    countries = df['country'].unique()
    t_stats = []
    for c in countries:
        sub = df[df['country'] == c][col].dropna()
        if len(sub) < min_obs:
            continue
        try:
            result = adfuller(sub, autolag='AIC', maxlag=2)
            t_stats.append(result[0])
        except Exception:
            continue

    N = len(t_stats)
    if N == 0:
        print(f"No valid series for {col}")
        return

    t_bar = np.mean(t_stats)
    # Approximate E[t_ADF | H0, T≈30] and sd
    mu_bar = -1.52
    sigma_bar = 0.89
    W = np.sqrt(N) * (t_bar - mu_bar) / sigma_bar
    p_val = stats.norm.cdf(W)  # one-sided: reject if W << 0

    print(f"\nIPS unit root test for '{col}':")
    print(f"  N countries = {N}, mean ADF t-stat = {t_bar:.3f}")
    print(f"  W statistic = {W:.3f}, p-value = {p_val:.4f} (H0: unit root)")
    pct_reject = sum(1 for t in t_stats if t < -2.86) / N
    print(f"  % countries rejecting ADF at 5% = {pct_reject:.1%}")
    if p_val < 0.05:
        print("  REJECT H0: panel appears stationary (I(0)) — TWFE valid")
    else:
        print("  CANNOT REJECT H0: unit root possible — interpret TWFE with caution")
    return t_bar, W, p_val

ips_unit_root(panel, 'inflation')
ips_unit_root(panel, 'm2_growth')



IPS unit root test for 'inflation':
  N countries = 160, mean ADF t-stat = -3.940
  W statistic = -34.398, p-value = 0.0000 (H0: unit root)
  % countries rejecting ADF at 5% = 64.4%
  REJECT H0: panel appears stationary (I(0)) — TWFE valid



IPS unit root test for 'm2_growth':
  N countries = 160, mean ADF t-stat = -4.074
  W statistic = -36.298, p-value = 0.0000 (H0: unit root)
  % countries rejecting ADF at 5% = 76.9%
  REJECT H0: panel appears stationary (I(0)) — TWFE valid


(-4.073958028473943, -36.29808727393631, 8.671665505556224e-289)

In [10]:
# Driscoll-Kraay SEs: robust to cross-sectional dependence and serial correlation
# Implemented via HAC kernel correction in linearmodels PanelOLS
from linearmodels.panel import PanelOLS

# Full sample DK SEs
panel_idx = panel.set_index(['country', 'year'])
panel_idx_clean = panel_idx.dropna(subset=['inflation', 'm2_growth'])

mod_dk = PanelOLS.from_formula(
    'inflation ~ 1 + m2_growth + EntityEffects + TimeEffects',
    data=panel_idx_clean
)
res_dk_full = mod_dk.fit(cov_type='kernel', kernel='bartlett', bandwidth=4)
print("=== TWFE Full Sample — Driscoll-Kraay (HAC kernel, bandwidth=4) ===")
print(f"  m2_growth: coef={res_dk_full.params['m2_growth']:.4f}, "
      f"t={res_dk_full.tstats['m2_growth']:.3f}, "
      f"p={res_dk_full.pvalues['m2_growth']:.4f}")
print(f"  N obs: {res_dk_full.nobs}, within R2: {res_dk_full.rsquared_within:.4f}")

# Clean sample DK SEs
panel_clean = panel[panel['sample_low_inflation'] == 1].copy()
panel_clean_idx = panel_clean.set_index(['country', 'year']).dropna(subset=['inflation', 'm2_growth'])
mod_dk_clean = PanelOLS.from_formula(
    'inflation ~ 1 + m2_growth + EntityEffects + TimeEffects',
    data=panel_clean_idx
)
res_dk_clean = mod_dk_clean.fit(cov_type='kernel', kernel='bartlett', bandwidth=4)
print("\n=== TWFE Clean Sample — Driscoll-Kraay ===")
print(f"  m2_growth: coef={res_dk_clean.params['m2_growth']:.4f}, "
      f"t={res_dk_clean.tstats['m2_growth']:.3f}, "
      f"p={res_dk_clean.pvalues['m2_growth']:.4f}")
print(f"  N obs: {res_dk_clean.nobs}, within R2: {res_dk_clean.rsquared_within:.4f}")

print("\nNote: DK SEs are robust to cross-sectional dependence and serial correlation.")
print("Compare to clustered SEs above — wider DK SEs indicate residual CSD.")


=== TWFE Full Sample — Driscoll-Kraay (HAC kernel, bandwidth=4) ===
  m2_growth: coef=0.6649, t=2.945, p=0.0033
  N obs: 4750, within R2: 0.5041

=== TWFE Clean Sample — Driscoll-Kraay ===
  m2_growth: coef=0.0403, t=1.456, p=0.1454
  N obs: 3639, within R2: 0.0292

Note: DK SEs are robust to cross-sectional dependence and serial correlation.
Compare to clustered SEs above — wider DK SEs indicate residual CSD.


In [11]:
# COVID robustness: exclude 2020-2021 to test structural break sensitivity
panel_no_covid = panel[~panel['year'].isin([2020, 2021])].copy()
panel_no_covid_idx = panel_no_covid.set_index(['country', 'year']).dropna(subset=['inflation', 'm2_growth'])

mod_nocovid = PanelOLS.from_formula(
    'inflation ~ 1 + m2_growth + EntityEffects + TimeEffects',
    data=panel_no_covid_idx
)
res_nocovid = mod_nocovid.fit(cov_type='clustered', cluster_entity=True)

panel_clean_no_covid = panel_no_covid[panel_no_covid['sample_low_inflation'] == 1].copy()
panel_clean_no_covid_idx = panel_clean_no_covid.set_index(['country', 'year']).dropna(subset=['inflation', 'm2_growth'])
mod_nocovid_clean = PanelOLS.from_formula(
    'inflation ~ 1 + m2_growth + EntityEffects + TimeEffects',
    data=panel_clean_no_covid_idx
)
res_nocovid_clean = mod_nocovid_clean.fit(cov_type='clustered', cluster_entity=True)

print("=== COVID Robustness: Exclude 2020-2021 ===")
print(f"Full sample (no COVID): coef={res_nocovid.params['m2_growth']:.4f}, p={res_nocovid.pvalues['m2_growth']:.4f}, n={res_nocovid.nobs}")
print(f"Clean sample (no COVID): coef={res_nocovid_clean.params['m2_growth']:.4f}, p={res_nocovid_clean.pvalues['m2_growth']:.4f}, n={res_nocovid_clean.nobs}")
print("\nWith COVID full baseline: coef=0.6649 (FE table above)")
print("If coefficients change materially, COVID years drive the result.")


=== COVID Robustness: Exclude 2020-2021 ===
Full sample (no COVID): coef=0.6641, p=0.0002, n=4479
Clean sample (no COVID): coef=0.0405, p=0.0667, n=3431

With COVID full baseline: coef=0.6649 (FE table above)
If coefficients change materially, COVID years drive the result.


In [12]:
# Sample mismatch test: Obj A uses >=30 obs countries; TWFE uses 160 countries.
# If TWFE on the >=30 subset gives the same result, the 160-country TWFE is consistent.
import statsmodels.api as sm

country_obs = panel.groupby('country')[['m2_growth', 'inflation']].count().min(axis=1)
countries_30obs = country_obs[country_obs >= 30].index.tolist()
print(f"Countries with >=30 obs (Obj A set): n={len(countries_30obs)}")

panel_30 = panel[panel['country'].isin(countries_30obs)].copy()
panel_30_idx = panel_30.set_index(['country', 'year']).dropna(subset=['inflation', 'm2_growth'])

mod_30 = PanelOLS.from_formula(
    'inflation ~ 1 + m2_growth + EntityEffects + TimeEffects',
    data=panel_30_idx
)
res_30 = mod_30.fit(cov_type='clustered', cluster_entity=True)

print(f"\n=== TWFE on {len(countries_30obs)}-country sample (same as Obj A) ===")
print(f"  m2_growth: coef={res_30.params['m2_growth']:.4f}, p={res_30.pvalues['m2_growth']:.4f}")
print(f"  N obs: {res_30.nobs}, within R2: {res_30.rsquared_within:.4f}")
print(f"\nCompare: TWFE full (160c) coef=0.6649; if subset is similar, wedge is real.")

# Between estimator (Obj A style) on all 160 countries
country_means_all = panel.groupby('country')[['m2_growth', 'inflation']].mean().dropna()
X_all = sm.add_constant(country_means_all['m2_growth'])
y_all = country_means_all['inflation']
res_objA_160 = sm.OLS(y_all, X_all).fit(cov_type='HC1')
print(f"\n=== Obj A (between estimator) on all 160 countries ===")
print(f"  m2_growth slope: {res_objA_160.params['m2_growth']:.4f}, p={res_objA_160.pvalues['m2_growth']:.4f}, R2={res_objA_160.rsquared:.4f}")
print(f"  Compare to Obj A restricted (>=30 obs subset): slope~0.952")


Countries with >=30 obs (Obj A set): n=108

=== TWFE on 108-country sample (same as Obj A) ===
  m2_growth: coef=0.8202, p=0.0000
  N obs: 3583, within R2: 0.5741

Compare: TWFE full (160c) coef=0.6649; if subset is similar, wedge is real.

=== Obj A (between estimator) on all 160 countries ===
  m2_growth slope: 0.8789, p=0.0000, R2=0.8507
  Compare to Obj A restricted (>=30 obs subset): slope~0.952
